# Train weighted joint KIC flow

Each `flow_id` encodes the x variable and weight/query choice. Select flows with `FLOW_IDS_TO_RUN`; saved models are used directly by `run_alpha_inference.ipynb`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from jax import random

cwd = Path.cwd()
LIKELIHOOD_DIR = cwd if cwd.name == "likelihood_wkic_joint" else cwd / "likelihood_wkic_joint"
REPO_ROOT = LIKELIHOOD_DIR.parent
for path in (REPO_ROOT, LIKELIHOOD_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from likelihood_wkic_common.flowutils import (
    add_standard_columns,
    catalog_to_theta,
    save_flow_theta,
    train_flow_theta,
    write_metadata,
)
from likelihood_wkic_common.flow_diagnostics import (
    library_versions,
    run_joint_flow_diagnostics,
)


In [2]:
KIC_DATA_DIR = REPO_ROOT / "pdet"
SEED = 0
VALIDATION_FRACTION = 0.
VALIDATION_SEED = 123
DIAGNOSTIC_SEED = 123
DIAGNOSTIC_N_POINTS = 8000

COMMON_TRAIN_KWARGS = dict(
    learning_rate=3e-4,
    max_epochs=500,
    max_patience=50,
    batch_size=1024,
    knots=16,
    interval=8,
)

FLOW_SPECS = {
    "m15_kic_pdet_teff5_all": {
        "description": "KIC sample weighted by pdet; Teff/logr/logp/lograd/kepmag.",
        "data_path": KIC_DATA_DIR / "m15_kic_w_pdet_TR.csv",
        "keys": ["Teff", "logr", "logp", "lograd", "kepmag"],
        "weight_col": "pdet",
        "query": None,
        "train_kwargs": {},
    },
    "m15_kic_pdet_teff5_pshort": {
        "description": "KIC sample weighted by pdet; Teff/logr/logp/lograd/kepmag; short-period KOI weights.",
        "data_path": KIC_DATA_DIR / "m15_kic_w_pdet_TR_p-short.csv",
        "keys": ["Teff", "logr", "logp", "lograd", "kepmag"],
        "weight_col": "pdet",
        "query": None,
        "train_kwargs": {},
    },
    "m15_kic_pdet_teff5_plong": {
        "description": "KIC sample weighted by pdet; Teff/logr/logp/lograd/kepmag; long-period KOI weights.",
        "data_path": KIC_DATA_DIR / "m15_kic_w_pdet_TR_p-long.csv",
        "keys": ["Teff", "logr", "logp", "lograd", "kepmag"],
        "weight_col": "pdet",
        "query": None,
        "train_kwargs": {},
    },
    "m15_kic_pdet_teff2_all": {
        "description": "KIC sample weighted by pdet; Teff/logr only.",
        "data_path": KIC_DATA_DIR / "m15_kic_w_pdet_TR.csv",
        "keys": ["Teff", "logr"],
        "weight_col": "pdet",
        "query": None,
        "train_kwargs": {},
    },
    "kic_pdet_rossby5_all": {
        "description": "KIC sample weighted by pdet; logRo/logr/logp/lograd/kepmag.",
        "data_path": KIC_DATA_DIR / "m15_kic_w_pdet_TR.csv",
        "keys": ["logRo", "logr", "logp", "lograd", "kepmag"],
        "weight_col": "pdet",
        "query": None,
        "train_kwargs": {},
    },
    "kic_pdet_rossby5_pshort": {
        "description": "KIC sample weighted by pdet; logRo/logr/logp/lograd/kepmag; short-period KOI weights.",
        "data_path": KIC_DATA_DIR / "m15_kic_w_pdet_TR_p-short.csv",
        "keys": ["logRo", "logr", "logp", "lograd", "kepmag"],
        "weight_col": "pdet",
        "query": None,
        "train_kwargs": {},
    },
    "kic_pdet_rossby5_plong": {
        "description": "KIC sample weighted by pdet; logRo/logr/logp/lograd/kepmag; long-period KOI weights.",
        "data_path": KIC_DATA_DIR / "m15_kic_w_pdet_TR_p-long.csv",
        "keys": ["logRo", "logr", "logp", "lograd", "kepmag"],
        "weight_col": "pdet",
        "query": None,
        "train_kwargs": {},
    },
}

FLOW_IDS_TO_RUN = [
    "m15_kic_pdet_teff2_all",
    "m15_kic_pdet_teff5_all",
    "m15_kic_pdet_teff5_pshort",
    "m15_kic_pdet_teff5_plong",
    "kic_pdet_rossby5_all",
    "kic_pdet_rossby5_pshort",
    "kic_pdet_rossby5_plong",
]
FLOW_IDS_TO_RUN


['m15_kic_pdet_teff2_all',
 'm15_kic_pdet_teff5_all',
 'm15_kic_pdet_teff5_pshort',
 'm15_kic_pdet_teff5_plong',
 'kic_pdet_rossby5_all',
 'kic_pdet_rossby5_pshort',
 'kic_pdet_rossby5_plong']

In [3]:
def split_fit_validation(df, *, fraction, seed):
    if fraction <= 0.0 or len(df) < 2:
        return df.reset_index(drop=True), None
    rng = np.random.default_rng(seed)
    validation_mask = rng.random(len(df)) < fraction
    if validation_mask.all():
        validation_mask[rng.integers(len(validation_mask))] = False
    if not validation_mask.any():
        validation_mask[rng.integers(len(validation_mask))] = True
    return (
        df.loc[~validation_mask].reset_index(drop=True),
        df.loc[validation_mask].reset_index(drop=True),
    )


def prepare_training_data(flow_id, spec):
    keys = list(spec["keys"])
    weight_col = spec.get("weight_col")
    df_input = add_standard_columns(pd.read_csv(spec["data_path"]))
    if spec.get("query"):
        df_input = df_input.query(spec["query"]).copy()
    required = list(dict.fromkeys(keys + ([] if weight_col is None else [weight_col])))
    df_clean = (
        df_input.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=required)
        .reset_index(drop=True)
    )
    df_fit, df_validation = split_fit_validation(
        df_clean,
        fraction=VALIDATION_FRACTION,
        seed=VALIDATION_SEED,
    )
    if len(df_fit) == 0:
        raise ValueError(f"No KIC rows remain for training {flow_id}.")
    theta_fit = catalog_to_theta(df_fit, keys)
    weights_fit = None if weight_col is None else df_fit[weight_col].to_numpy(float)
    theta_validation = None if df_validation is None else catalog_to_theta(df_validation, keys)
    weights_validation = None if df_validation is None or weight_col is None else df_validation[weight_col].to_numpy(float)
    return {
        "df_input": df_input,
        "df_clean": df_clean,
        "df_fit": df_fit,
        "df_validation": df_validation,
        "theta_fit": theta_fit,
        "weights_fit": weights_fit,
        "theta_validation": theta_validation,
        "weights_validation": weights_validation,
    }


for flow_id in FLOW_IDS_TO_RUN:
    data = prepare_training_data(flow_id, FLOW_SPECS[flow_id])
    print(flow_id)
    print(f"  input rows: {len(data['df_input'])}")
    print(f"  clean rows: {len(data['df_clean'])}")
    print(f"  fit rows: {len(data['df_fit'])}")
    print(f"  validation rows: {0 if data['df_validation'] is None else len(data['df_validation'])}")
    print(f"  keys: {FLOW_SPECS[flow_id]['keys']}")
    print(f"  data: {FLOW_SPECS[flow_id]['data_path']}")


m15_kic_pdet_teff2_all
  input rows: 29700
  clean rows: 29700
  fit rows: 29700
  validation rows: 0
  keys: ['Teff', 'logr']
  data: <repo>/pdet/m15_kic_w_pdet_TR.csv
m15_kic_pdet_teff5_all
  input rows: 29700
  clean rows: 29700
  fit rows: 29700
  validation rows: 0
  keys: ['Teff', 'logr', 'logp', 'lograd', 'kepmag']
  data: <repo>/pdet/m15_kic_w_pdet_TR.csv
m15_kic_pdet_teff5_pshort
  input rows: 29700
  clean rows: 29700
  fit rows: 29700
  validation rows: 0
  keys: ['Teff', 'logr', 'logp', 'lograd', 'kepmag']
  data: <repo>/pdet/m15_kic_w_pdet_TR_p-short.csv
m15_kic_pdet_teff5_plong
  input rows: 29700
  clean rows: 29700
  fit rows: 29700
  validation rows: 0
  keys: ['Teff', 'logr', 'logp', 'lograd', 'kepmag']
  data: <repo>/pdet/m15_kic_w_pdet_TR_p-long.csv
kic_pdet_rossby5_all
  input rows: 29700
  clean rows: 29700
  fit rows: 29700
  validation rows: 0
  keys: ['logRo', 'logr', 'logp', 'lograd', 'kepmag']
  data: <repo>/pdet/m15_kic_w_pdet_TR.csv
kic_pdet_rossby5_pshort


In [4]:
runs = {}

for i, flow_id in enumerate(FLOW_IDS_TO_RUN):
    spec = FLOW_SPECS[flow_id]
    data = prepare_training_data(flow_id, spec)
    train_kwargs = dict(COMMON_TRAIN_KWARGS)
    train_kwargs.update(spec.get("train_kwargs", {}))

    flow, losses, _draws, meta = train_flow_theta(
        data["theta_fit"],
        weights=data["weights_fit"],
        key=random.key(SEED),
        keys=spec["keys"],
        weight_col=spec.get("weight_col"),
        **train_kwargs,
    )

    flow_dir = LIKELIHOOD_DIR / "flows" / flow_id
    model_path = flow_dir / "model.eqx"
    meta.update(
        dict(
            flow_id=flow_id,
            flow_dir=flow_dir,
            data_path=spec["data_path"],
            description=spec["description"],
            query=spec.get("query"),
            seed=SEED,
            n_input=len(data["df_input"]),
            n_clean=len(data["df_clean"]),
            n_train=len(data["df_fit"]),
            n_validation=0 if data["df_validation"] is None else len(data["df_validation"]),
            validation_fraction=VALIDATION_FRACTION,
            validation_seed=VALIDATION_SEED,
            library_versions=library_versions(),
        )
    )

    save_flow_theta(model_path, flow, meta)
    write_metadata(flow_dir / "metadata.json", meta)
    diagnostics = run_joint_flow_diagnostics(
        flow,
        data["theta_fit"],
        meta,
        flow_dir,
        losses=losses,
        weights_fit=data["weights_fit"],
        theta_validation=data["theta_validation"],
        weights_validation=data["weights_validation"],
        seed=DIAGNOSTIC_SEED,
        n_points=DIAGNOSTIC_N_POINTS,
    )
    runs[flow_id] = {"model_path": model_path, "diagnostics": diagnostics["paths"]}
    print(f"saved {flow_id} -> {flow_dir}")

runs


 40%|███▉      | 198/500 [01:12<01:51,  2.72it/s, train=2.57, val=2.54 (Max patience reached)]


saved m15_kic_pdet_teff2_all -> <repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff2_all


 27%|██▋       | 136/500 [01:12<03:15,  1.86it/s, train=4.22, val=4.24 (Max patience reached)]


saved m15_kic_pdet_teff5_all -> <repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff5_all


 27%|██▋       | 136/500 [01:12<03:14,  1.87it/s, train=4.28, val=4.28 (Max patience reached)]


...

 27%|██▋       | 136/500 [01:14<03:19,  1.82it/s, train=4, val=3.98 (Max patience reached)]


saved kic_pdet_rossby5_pshort -> <repo>/likelihood_wkic_joint/flows/kic_pdet_rossby5_pshort


 27%|██▋       | 136/500 [01:10<03:09,  1.92it/s, train=3.88, val=3.89 (Max patience reached)]


saved kic_pdet_rossby5_plong -> <repo>/likelihood_wkic_joint/flows/kic_pdet_rossby5_plong


{'m15_kic_pdet_teff2_all': {'model_path': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff2_all/model.eqx'),
  'diagnostics': {'logprob_summary': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff2_all/logprob_summary.csv'),
   'loss_curve': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff2_all/loss_curve.png'),
   'check_corner': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff2_all/check_corner.png'),
   'diagnostics': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff2_all/diagnostics.json')}},
 'm15_kic_pdet_teff5_all': {'model_path': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff5_all/model.eqx'),
  'diagnostics': {'logprob_summary': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff5_all/logprob_summary.csv'),
   'loss_curve': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_kic_pdet_teff5_all/loss_curve.png'),
   'check_corner': PosixPath('<repo>/likelihood_wkic_joint/flows/m15_